In [0]:
USE CATALOG smart_odoo;

-- =====================================================
-- Gold Fact Purchase Order
-- =====================================================

CREATE TABLE IF NOT EXISTS gold.fact_purchase_order
(
    purchase_order_line_id INT,

    purchase_order_id INT,

    supplier_id INT,
    company_id INT,

    picking_type_id INT,
    order_user_id INT,

    product_id INT,
    product_uom_id INT,

    order_state STRING,
    invoice_status STRING,

    amount_untaxed DOUBLE,
    amount_tax DOUBLE,
    amount_total DOUBLE,

    product_qty DOUBLE,
    product_uom_qty DOUBLE,

    price_unit DOUBLE,
    price_subtotal DOUBLE,
    price_total DOUBLE,

    qty_invoiced DOUBLE,
    qty_received DOUBLE,
    qty_to_invoice DOUBLE,

    discount DOUBLE,

    line_description STRING,

    order_date DATE,
    date_approve DATE,
    date_planned DATE,

    order_created_at TIMESTAMP,
    order_updated_at TIMESTAMP,

    dwh_loaded_at TIMESTAMP
)
USING DELTA;

-- =====================================================
-- MERGE
-- =====================================================

MERGE INTO gold.fact_purchase_order AS target

USING
(
    SELECT
        TRY_CAST(pol.purchase_order_line_id AS INT) AS purchase_order_line_id,

        TRY_CAST(po.purchase_order_id AS INT) AS purchase_order_id,

        TRY_CAST(po.partner_id AS INT) AS supplier_id,
        TRY_CAST(po.company_id AS INT) AS company_id,

        TRY_CAST(po.picking_type_id AS INT) AS picking_type_id,
        TRY_CAST(po.user_id AS INT) AS order_user_id,

        TRY_CAST(pol.product_id AS INT) AS product_id,
        TRY_CAST(pol.product_uom_id AS INT) AS product_uom_id,

        po.order_state,
        po.invoice_status,

        CAST(po.amount_untaxed AS DOUBLE) AS amount_untaxed,
        CAST(po.amount_tax AS DOUBLE) AS amount_tax,
        CAST(po.amount_total AS DOUBLE) AS amount_total,

        CAST(pol.product_qty AS DOUBLE) AS product_qty,
        CAST(pol.product_uom_qty AS DOUBLE) AS product_uom_qty,

        CAST(pol.price_unit AS DOUBLE) AS price_unit,
        CAST(pol.price_subtotal AS DOUBLE) AS price_subtotal,
        CAST(pol.price_total AS DOUBLE) AS price_total,

        CAST(pol.qty_invoiced AS DOUBLE) AS qty_invoiced,
        CAST(pol.qty_received AS DOUBLE) AS qty_received,
        CAST(pol.qty_to_invoice AS DOUBLE) AS qty_to_invoice,

        CAST(pol.discount AS DOUBLE) AS discount,

        pol.line_description,

        CAST(po.date_order AS DATE) AS order_date,
        CAST(po.date_approve AS DATE) AS date_approve,
        CAST(po.date_planned AS DATE) AS date_planned,

        po.created_at AS order_created_at,
        po.updated_at AS order_updated_at,

        current_timestamp() AS dwh_loaded_at

    FROM silver.purchase_order po

    LEFT JOIN silver.purchase_order_line pol
        ON pol.order_id = po.purchase_order_id

    WHERE po.updated_at >=
    (
        SELECT COALESCE(
            MAX(order_updated_at) - INTERVAL 1 DAY,
            TIMESTAMP('1900-01-01')
        )
        FROM gold.fact_purchase_order
    )

) AS source

ON target.purchase_order_line_id = source.purchase_order_line_id

WHEN MATCHED
AND target.order_updated_at < source.order_updated_at

THEN UPDATE SET

    target.purchase_order_id = source.purchase_order_id,

    target.supplier_id = source.supplier_id,
    target.company_id = source.company_id,

    target.picking_type_id = source.picking_type_id,
    target.order_user_id = source.order_user_id,

    target.product_id = source.product_id,
    target.product_uom_id = source.product_uom_id,

    target.order_state = source.order_state,
    target.invoice_status = source.invoice_status,

    target.amount_untaxed = source.amount_untaxed,
    target.amount_tax = source.amount_tax,
    target.amount_total = source.amount_total,

    target.product_qty = source.product_qty,
    target.product_uom_qty = source.product_uom_qty,

    target.price_unit = source.price_unit,
    target.price_subtotal = source.price_subtotal,
    target.price_total = source.price_total,

    target.qty_invoiced = source.qty_invoiced,
    target.qty_received = source.qty_received,
    target.qty_to_invoice = source.qty_to_invoice,

    target.discount = source.discount,

    target.line_description = source.line_description,

    target.order_date = source.order_date,
    target.date_approve = source.date_approve,
    target.date_planned = source.date_planned,

    target.order_created_at = source.order_created_at,
    target.order_updated_at = source.order_updated_at,

    target.dwh_loaded_at = current_timestamp()

WHEN NOT MATCHED THEN

INSERT
(
    purchase_order_line_id,
    purchase_order_id,
    supplier_id,
    company_id,
    picking_type_id,
    order_user_id,
    product_id,
    product_uom_id,
    order_state,
    invoice_status,
    amount_untaxed,
    amount_tax,
    amount_total,
    product_qty,
    product_uom_qty,
    price_unit,
    price_subtotal,
    price_total,
    qty_invoiced,
    qty_received,
    qty_to_invoice,
    discount,
    line_description,
    order_date,
    date_approve,
    date_planned,
    order_created_at,
    order_updated_at,
    dwh_loaded_at
)

VALUES
(
    source.purchase_order_line_id,
    source.purchase_order_id,
    source.supplier_id,
    source.company_id,
    source.picking_type_id,
    source.order_user_id,
    source.product_id,
    source.product_uom_id,
    source.order_state,
    source.invoice_status,
    source.amount_untaxed,
    source.amount_tax,
    source.amount_total,
    source.product_qty,
    source.product_uom_qty,
    source.price_unit,
    source.price_subtotal,
    source.price_total,
    source.qty_invoiced,
    source.qty_received,
    source.qty_to_invoice,
    source.discount,
    source.line_description,
    source.order_date,
    source.date_approve,
    source.date_planned,
    source.order_created_at,
    source.order_updated_at,
    current_timestamp()
);